# Android SMS Classifier — 一键训练（VS Code / Colab）

教师模型已改为 **PyTorch**（`transformers` 5 起不再提供 `TFAutoModel*`）。
学生蒸馏/剪枝/量化仍用 TensorFlow。

## 安全

- **不要把 GitHub Token 写进本文件或提交到 Git。**
- Colab 私有库：用 Colab Secrets / 环境变量注入 token，或临时在配置格填写后立刻撤销。
- 禁止上传真实短信。

## 用法

1. 选内核（本机 `.venv` 或 Colab GPU）
2. 改配置格（Colab 填 `GIT_URL`，**不要**把 token 提交进仓库）
3. **Run All**

若刚修过代码：先把本仓库最新改动 **push 到远端**，再在 Colab 跑（否则 clone 仍是旧的 TF 教师脚本）。

## 0. 配置

In [ ]:
# ========= 用户配置 =========
MODE = "auto"  # auto | local | colab

# Colab 专用。公开库直接填 HTTPS；私有库用下面 Secrets 方式，勿把 token 写死在这里。
GIT_URL = ""  # 例: "https://github.com/ORG/Android_SMS_Classifier.git"
GIT_BRANCH = "main"

# 若用 Colab Secrets：先在 Colab 添加 GH_TOKEN，再取消下一行注释
# from google.colab import userdata
# GIT_URL = f"https://{userdata.get('GH_TOKEN')}@github.com/ORG/Android_SMS_Classifier.git"

COLAB_WORKDIR = "/content/Android_SMS_Classifier"
REGENERATE_SYNTHETIC = True  # Colab 无 processed 数据时建议 True
PER_LABEL_LANG = 40
TEACHER_MAX_SAMPLES = 0
SEED = 42
BERT_MODEL_ID = "google-bert/bert-base-multilingual-cased"  # 【第三方】
# ============================

## 1. 定位项目根目录

In [ ]:
from __future__ import annotations

import os
import shutil
import subprocess
import sys
from pathlib import Path


def running_in_colab() -> bool:
    try:
        import google.colab  # noqa: F401

        return True
    except ImportError:
        return False


def find_repo_root(start: Path) -> Path | None:
    cur = start.resolve()
    for p in [cur, *cur.parents]:
        if (p / "training" / "scripts" / "train_teacher.py").is_file():
            return p
    return None


def clone_repo(url: str, branch: str, dest: Path) -> Path:
    if not url:
        raise ValueError(
            "Colab 模式请填写 GIT_URL（不要把 token 提交进 Git）。"
        )
    if dest.exists():
        if (dest / ".git").exists():
            subprocess.run(["git", "-C", str(dest), "fetch", "--all"], check=False)
            subprocess.run(["git", "-C", str(dest), "checkout", branch], check=False)
            subprocess.run(
                ["git", "-C", str(dest), "pull", "--ff-only", "origin", branch],
                check=False,
            )
            return dest
        shutil.rmtree(dest)
    dest.parent.mkdir(parents=True, exist_ok=True)
    subprocess.run(
        ["git", "clone", "--branch", branch, "--depth", "1", url, str(dest)],
        check=True,
    )
    return dest


IN_COLAB = running_in_colab()
effective_mode = MODE
if MODE == "auto":
    effective_mode = "colab" if IN_COLAB else "local"

print(f"IN_COLAB={IN_COLAB} MODE={MODE} -> {effective_mode}")

if effective_mode == "colab":
    PROJECT_ROOT = clone_repo(GIT_URL, GIT_BRANCH, Path(COLAB_WORKDIR))
    BERT_CACHE = Path("/content/hf_cache/bert-base-multilingual-cased")
else:
    PROJECT_ROOT = find_repo_root(Path.cwd())
    if PROJECT_ROOT is None:
        here = Path.cwd()
        for cand in [here, here / "Android_SMS_Classifier", here.parent]:
            PROJECT_ROOT = find_repo_root(cand)
            if PROJECT_ROOT is not None:
                break
    if PROJECT_ROOT is None:
        raise FileNotFoundError("找不到仓库根目录，请用 VS Code 打开本仓库后再运行。")
    BERT_CACHE = PROJECT_ROOT / "training" / ".cache" / "bert-base-multilingual-cased"

os.chdir(PROJECT_ROOT)
os.environ["PYTHONPATH"] = str(PROJECT_ROOT / "training")
sys.path.insert(0, str(PROJECT_ROOT / "training"))

print("PROJECT_ROOT =", PROJECT_ROOT)
print("BERT_CACHE   =", BERT_CACHE)
try:
    print(subprocess.check_output(["nvidia-smi", "-L"], text=True).strip())
except Exception as exc:  # noqa: BLE001
    print("nvidia-smi:", exc)
print("Python", sys.version)

## 2. 安装依赖（教师用 PyTorch，学生用 TensorFlow）

若提示需要重启 runtime：Runtime → Restart session，然后从第 1 格重新跑（可跳过已装好的 pip）。

In [ ]:
import subprocess
import sys
from pathlib import Path


def pip_install(args: list[str]) -> None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", *args])


pip_install(["-U", "pip"])

# 显式安装：避免 Colab 自带 transformers 5 却按旧 TF API 用
pip_install(
    [
        "-q",
        "torch",
        "transformers",
        "tensorflow>=2.16",
        "tensorflow-model-optimization",
        "scikit-learn",
        "PyYAML",
        "numpy",
    ]
)

train_req = Path("training/requirements-train.txt")
if train_req.exists():
    try:
        pip_install(["-q", "-r", str(train_req)])
    except subprocess.CalledProcessError as exc:
        print("requirements-train 部分失败，已用上面精简依赖继续:", exc)

import torch
import transformers
import tensorflow as tf
from transformers import AutoModelForSequenceClassification

print("torch", torch.__version__, "cuda", torch.cuda.is_available())
print("transformers", transformers.__version__)
print("TF", tf.__version__, "GPU", tf.config.list_physical_devices("GPU"))
print("AutoModelForSequenceClassification OK:", AutoModelForSequenceClassification.__name__)
if not torch.cuda.is_available():
    print("WARNING: PyTorch 未检测到 CUDA。Colab 请确认内核为 GPU。")

## 3. 缓存教师 BERT（PyTorch 权重）【第三方】

In [ ]:
from pathlib import Path
import hashlib
from transformers import AutoModelForSequenceClassification, AutoTokenizer

BERT_CACHE.mkdir(parents=True, exist_ok=True)
marker = BERT_CACHE / "config.json"
weights_ok = any(BERT_CACHE.glob("model.safetensors")) or any(BERT_CACHE.glob("pytorch_model.bin"))

if marker.exists() and weights_ok:
    print("Reuse existing BERT cache:", BERT_CACHE)
else:
    print("Downloading", BERT_MODEL_ID, "->", BERT_CACHE)
    tok = AutoTokenizer.from_pretrained(BERT_MODEL_ID)
    mdl = AutoModelForSequenceClassification.from_pretrained(BERT_MODEL_ID, num_labels=4)
    tok.save_pretrained(BERT_CACHE)
    mdl.save_pretrained(BERT_CACHE)
    print("Saved.")


def sha256_dir(path: Path) -> str:
    h = hashlib.sha256()
    for f in sorted(path.rglob("*")):
        if f.is_file():
            h.update(f.relative_to(path).as_posix().encode())
            h.update(f.read_bytes())
    return h.hexdigest()


print("BERT cache sha256:", sha256_dir(BERT_CACHE))

## 4. 数据准备与检查

In [ ]:
import json
import os
import subprocess
import sys
from collections import Counter
from pathlib import Path

os.environ["PYTHONPATH"] = str(PROJECT_ROOT / "training")


def run_py(*script_and_args: str) -> None:
    cmd = [sys.executable, *script_and_args]
    print("+", " ".join(cmd))
    subprocess.check_call(cmd, cwd=str(PROJECT_ROOT))


train_jsonl = PROJECT_ROOT / "training" / "data" / "processed" / "train.jsonl"
if REGENERATE_SYNTHETIC or not train_jsonl.exists():
    print("generating synthetic dataset…")
    run_py(
        "training/scripts/generate_synthetic_dataset.py",
        "--per-label-lang",
        str(PER_LABEL_LANG),
    )
    run_py("training/scripts/build_dataset.py", "--augment-train")
    run_py("training/scripts/build_adversarial_slices.py")
else:
    print("reuse existing processed data:", train_jsonl)

run_py("training/scripts/check_split_leakage.py")
run_py("training/scripts/validate_labels.py")

need = {"TRANSACTION", "AD", "HARASS", "FRAUD"}
print("\n=== split label counts ===")
for split in ["train", "validation", "test"]:
    p = PROJECT_ROOT / "training" / "data" / "processed" / f"{split}.jsonl"
    labels = [
        json.loads(line)["label"]
        for line in p.read_text(encoding="utf-8").splitlines()
        if line.strip()
    ]
    c = Counter(labels)
    print(f"{split}: n={len(labels)} {dict(c)}")
    missing = need - set(c)
    if missing and split != "train":
        print(f"WARNING: {split} missing {sorted(missing)}")

## 5. 微调教师 → 蒸馏 → 剪枝 → 量化 → 验证

In [ ]:
import json
import os
import subprocess
import sys
from pathlib import Path

os.environ["PYTHONPATH"] = str(PROJECT_ROOT / "training")


def run_py(*args: str) -> None:
    cmd = [sys.executable, *args]
    print("\n+", " ".join(cmd), flush=True)
    subprocess.check_call(cmd, cwd=str(PROJECT_ROOT))


teacher_cmd = [
    "training/scripts/train_teacher.py",
    "--model-path",
    str(BERT_CACHE),
    "--seed",
    str(SEED),
]
if TEACHER_MAX_SAMPLES and TEACHER_MAX_SAMPLES > 0:
    teacher_cmd += ["--max-samples", str(TEACHER_MAX_SAMPLES)]

run_py(*teacher_cmd)

logits_manifest = (
    PROJECT_ROOT / "training" / "data" / "manifests" / "teacher_logits_manifest.json"
)
assert logits_manifest.is_file(), "教师失败：缺少 teacher_logits_manifest.json"

run_py("training/scripts/distill_student.py", "--seed", str(SEED))
distill = json.loads(
    (PROJECT_ROOT / "training" / "artifacts" / "student" / "distill_manifest.json").read_text(
        encoding="utf-8"
    )
)
print("distill used_distillation =", distill.get("used_distillation"))
assert distill.get("used_distillation") is True, "未真蒸馏，已中止"

run_py("training/scripts/prune_channels.py", "--seed", str(SEED))
run_py("training/scripts/quantize_int8.py", "--seed", str(SEED))
run_py("training/scripts/verify_tflite.py", "--seed", str(SEED))
run_py("training/scripts/evaluate.py", "--mode", "tflite", "--seed", str(SEED))

tflite = PROJECT_ROOT / "training" / "artifacts" / "student" / "sms_bytecnn_int8.tflite"
assert tflite.is_file(), f"缺少 {tflite}"
print("\nOK pipeline done:", tflite)

## 6. 导出

In [ ]:
import os
import shutil
import subprocess
import sys
import zipfile
from pathlib import Path

os.environ["PYTHONPATH"] = str(PROJECT_ROOT / "training")

if not IN_COLAB:
    subprocess.check_call(
        [sys.executable, "training/scripts/export_android_assets.py"],
        cwd=str(PROJECT_ROOT),
    )
    asset = (
        PROJECT_ROOT
        / "android"
        / "classifier-sdk"
        / "src"
        / "main"
        / "assets"
        / "model"
        / "sms_bytecnn_int8.tflite"
    )
    print("Exported:", asset, "exists=", asset.is_file())
else:
    export_dir = Path("/content/colab_export")
    if export_dir.exists():
        shutil.rmtree(export_dir)
    export_dir.mkdir(parents=True)
    shutil.copytree(PROJECT_ROOT / "training" / "artifacts", export_dir / "artifacts")
    metrics = PROJECT_ROOT / "training" / "reports" / "metrics"
    if metrics.exists():
        shutil.copytree(metrics, export_dir / "metrics")
    manifests = PROJECT_ROOT / "training" / "data" / "manifests"
    if manifests.exists():
        shutil.copytree(manifests, export_dir / "manifests")
    tflite = PROJECT_ROOT / "training" / "artifacts" / "student" / "sms_bytecnn_int8.tflite"
    if tflite.exists():
        shutil.copy2(tflite, export_dir / tflite.name)

    zip_path = Path("/content/colab_export.zip")
    if zip_path.exists():
        zip_path.unlink()
    with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
        for f in export_dir.rglob("*"):
            if f.is_file():
                zf.write(f, f.relative_to(export_dir.parent))
    print("Wrote", zip_path)
    try:
        from google.colab import files

        files.download(str(zip_path))
    except Exception as exc:  # noqa: BLE001
        print("自动下载失败，请手动下载:", zip_path, exc)